# Resolution vs Accuracy Experiment
**Model:** qwen/qwen3.5-122b-a10b via NVIDIA NIM  
**Purpose:** Measure the effect of image downsampling (None, 1024px, 768px) on NEDS and TEDS scores, and estimate token savings.  
**Sample:** 10 scanned + 10 photographed images per task (docs and tables).

In [ ]:
%pip install -U openai pillow pandas tqdm nltk

## Imports and Initialisation

In [5]:
import pandas as pd
import base64
import os
import json
import itertools
from io import BytesIO
from PIL import Image
from openai import OpenAI
from tqdm import tqdm

from doc_parsing_evaluator import ParsingEvaluator

# ── Paths ── update these to match your machine ──────────────────────────────
BASE_DIR = r"D:\Projects\Evaluation Of MultiModal LLMs for Layout Aware Document Parsing\CC-OCR_Dataset\doc_parsing"
OUT_DIR  = r"D:\Projects\Evaluation Of MultiModal LLMs for Layout Aware Document Parsing\Evaluation_Results\Qwen_ResolutionExperiment"
os.makedirs(OUT_DIR, exist_ok=True)

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME = "google/gemma-3-27b-it"

# ── API Keys ──────────────────────────────────────────────────────────────────
env_path = ".env"
if os.path.exists(env_path):
    with open(env_path, "r") as f:
        API_KEYS = [line.strip() for line in f if line.strip() and not line.startswith("#")]
else:
    API_KEYS = []

if API_KEYS:
    key_cycle = itertools.cycle(API_KEYS)
    print(f"Loaded {len(API_KEYS)} API key(s).")
else:
    key_cycle = None
    print("WARNING: No API keys found. Inference will fail.")

# ── Evaluator ─────────────────────────────────────────────────────────────────
evaluator = ParsingEvaluator(group_name="Resolution_Exp")

# ── Experiment Config ─────────────────────────────────────────────────────────
RESOLUTIONS  = [None, 1024, 768]   # None = original, integers = max edge in px
N_SAMPLES    = 10                  # images per split per resolution condition

print(f"Model      : {MODEL_NAME}")
print(f"Resolutions: {RESOLUTIONS}")
print(f"Samples    : {N_SAMPLES} per split per condition")

Loaded 1 API key(s).
Model      : google/gemma-3-27b-it
Resolutions: [None, 1024, 768]
Samples    : 10 per split per condition


## Image Pre-Processing Helper

In [6]:
def process_image(image_bytes, max_edge=None):
    """
    Resizes the image so its longest edge does not exceed max_edge pixels.
    If max_edge is None or the image is already small enough, no resize is done.
    Returns:
        resized_bytes : PNG bytes ready to encode as base64
        stats         : dict with original res, new res, token estimates, savings %
    """
    img = Image.open(BytesIO(image_bytes)).convert("RGB")
    orig_w, orig_h = img.size

    # Approximate token count: each 28x28 pixel patch = 1 visual token
    orig_tokens = (orig_w // 28) * (orig_h // 28)

    if max_edge and max(orig_w, orig_h) > max_edge:
        scale = max_edge / float(max(orig_w, orig_h))
        new_w = int(orig_w * scale)
        new_h = int(orig_h * scale)
        img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)

    new_w, new_h = img.size
    new_tokens = (new_w // 28) * (new_h // 28)
    savings_pct = round((1 - new_tokens / orig_tokens) * 100, 2) if orig_tokens > 0 else 0.0

    buf = BytesIO()
    img.save(buf, format="PNG")
    resized_bytes = buf.getvalue()

    stats = {
        "orig_res"         : f"{orig_w}x{orig_h}",
        "new_res"          : f"{new_w}x{new_h}",
        "orig_tokens"      : orig_tokens,
        "new_tokens"       : new_tokens,
        "token_savings_pct": savings_pct
    }
    return resized_bytes, stats

## Inference Engine

In [7]:
def query_nim(image_bytes, task_type):
    """Sends image bytes to the NIM API and returns the model's text response."""
    if key_cycle is None:
        return "Error: No API key"

    client = OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=next(key_cycle)
    )

    if task_type == "table":
        prompt = (
            "You are a Table Parsing assistant. Extract the table structure into clean HTML. "
            "Use common tags like <body>, <table>, <tr>, <td>, and <th> tags. "
            "Use rowspan/colspan only if cells are merged. "
            "Maintain the exact reading order of cells (left-to-right, row-by-row). "
            "Do NOT include any CSS or style attributes. Provide only the raw HTML code."
        )
    else:
        prompt = (
            "You are an expert Document Intelligence assistant. Extract text strictly in its logical reading order. "
            "Do NOT include document boilerplate (like \\documentclass). Use minimal LaTeX tags: "
            "\\section{...} for headers and \\textbf{...} for bold. Use LaTeX ($$) ONLY for math formulas. "
            "Output plain text with basic line breaks to reflect the flow. Do not attempt visual layout recreation."
        )

    image_b64 = base64.b64encode(image_bytes).decode("utf-8")

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": [
                {"type": "text",      "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}}
            ]}],
            max_tokens=2048,
            temperature=0.1
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

## Document Parsing Experiment

In [8]:
doc_tasks = {
    "Scanned": os.path.join(BASE_DIR, "doc", "doc_scan_eng_75.tsv"),
    "Photoed": os.path.join(BASE_DIR, "doc", "doc_photo_eng_75.tsv")
}

# Storage: doc_results[resolution_key][split_label] = avg_score
doc_results     = {str(r): {} for r in RESOLUTIONS}
doc_token_stats = {str(r): [] for r in RESOLUTIONS}  # per-image savings %
doc_res_stats   = {str(r): [] for r in RESOLUTIONS}  # per-image (orig_res, new_res, orig_tok, new_tok)

for label, path in doc_tasks.items():
    if not os.path.exists(path):
        print(f"WARNING: File not found, skipping {label}: {path}")
        continue

    df = pd.read_csv(path, sep="\t").head(N_SAMPLES)

    for res in RESOLUTIONS:
        res_key = str(res)
        scores, savings = [], []

        desc = f"Docs | {label} | {res if res else 'Original'}"
        for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
            raw_bytes = base64.b64decode(row["image"])
            gt        = str(row["answer"]).strip()

            proc_bytes, stats = process_image(raw_bytes, max_edge=res)
            doc_token_stats[res_key].append(stats["token_savings_pct"])
            doc_res_stats[res_key].append(stats)
            savings.append(stats["token_savings_pct"])

            pred  = query_nim(proc_bytes, "doc")
            score = evaluator.evaluate_single_doc_sample(gt, pred)
            scores.append(score)

        avg_score   = sum(scores)  / len(scores)  if scores  else 0
        avg_savings = sum(savings) / len(savings) if savings else 0
        doc_results[res_key][label] = avg_score

        print(f"  [{label} | {res if res else 'Original'}] "
              f"NEDS: {avg_score:.4f} | Avg Token Savings: {avg_savings:.1f}%")

print("\nDocument parsing experiment complete.")

Docs | Scanned | Original:  10%|█         | 1/10 [00:57<08:33, 57.11s/it]


KeyboardInterrupt: 

## Table Parsing Experiment

In [ ]:
table_tasks = {
    "Scanned": os.path.join(BASE_DIR, "table", "table_scan_eng_75.tsv"),
    "Photoed": os.path.join(BASE_DIR, "table", "table_photo_eng_75.tsv")
}

table_results     = {str(r): {} for r in RESOLUTIONS}
table_token_stats = {str(r): [] for r in RESOLUTIONS}
table_res_stats   = {str(r): [] for r in RESOLUTIONS}

for label, path in table_tasks.items():
    if not os.path.exists(path):
        print(f"WARNING: File not found, skipping {label}: {path}")
        continue

    df = pd.read_csv(path, sep="\t").head(N_SAMPLES)

    for res in RESOLUTIONS:
        res_key = str(res)
        scores, savings = [], []

        desc = f"Tables | {label} | {res if res else 'Original'}"
        for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
            raw_bytes = base64.b64decode(row["image"])
            gt        = str(row["answer"]).strip()

            proc_bytes, stats = process_image(raw_bytes, max_edge=res)
            table_token_stats[res_key].append(stats["token_savings_pct"])
            table_res_stats[res_key].append(stats)
            savings.append(stats["token_savings_pct"])

            pred  = query_nim(proc_bytes, "table")
            score = evaluator.evaluate_single_table_sample(gt, pred)
            scores.append(score)

        avg_score   = sum(scores)  / len(scores)  if scores  else 0
        avg_savings = sum(savings) / len(savings) if savings else 0
        table_results[res_key][label] = avg_score

        print(f"  [{label} | {res if res else 'Original'}] "
              f"TEDS: {avg_score:.4f} | Avg Token Savings: {avg_savings:.1f}%")

print("\nTable parsing experiment complete.")

## Final Report

In [ ]:
def avg(lst):
    return round(sum(lst) / len(lst), 4) if lst else 0.0

def avg_res_field(stats_list, field):
    """Average a numeric field across all per-image stat dicts."""
    vals = [s[field] for s in stats_list if isinstance(s[field], (int, float))]
    return round(sum(vals) / len(vals), 1) if vals else 0.0

report = {
    "Experiment" : f"Resolution vs Accuracy Tradeoff ({N_SAMPLES} Images per Split)",
    "Model"      : MODEL_NAME,
    "Document Parsing (NEDS)": doc_results,
    "Table Parsing (TEDS)"   : table_results,
    "Token Savings (Docs)": {
        str(r): avg(doc_token_stats[str(r)]) for r in RESOLUTIONS
    },
    "Token Savings (Tables)": {
        str(r): avg(table_token_stats[str(r)]) for r in RESOLUTIONS
    },
    "Resolution Stats (Docs)": {
        str(r): {
            "avg_orig_tokens" : avg_res_field(doc_res_stats[str(r)], "orig_tokens"),
            "avg_new_tokens"  : avg_res_field(doc_res_stats[str(r)], "new_tokens"),
            "avg_savings_pct" : avg(doc_token_stats[str(r)])
        } for r in RESOLUTIONS
    },
    "Resolution Stats (Tables)": {
        str(r): {
            "avg_orig_tokens" : avg_res_field(table_res_stats[str(r)], "orig_tokens"),
            "avg_new_tokens"  : avg_res_field(table_res_stats[str(r)], "new_tokens"),
            "avg_savings_pct" : avg(table_token_stats[str(r)])
        } for r in RESOLUTIONS
    }
}

report_path = os.path.join(OUT_DIR, "qwen_resolution_experiment_report.json")
with open(report_path, "w") as f:
    json.dump(report, f, indent=4)

print(f"Report saved to: {report_path}")
print("\n========= SUMMARY =========")
for r in RESOLUTIONS:
    rk = str(r)
    label = f"Max {r}px" if r else "Original"
    doc_sc  = doc_results[rk]
    tbl_sc  = table_results[rk]
    tok_sav = avg(doc_token_stats[rk] + table_token_stats[rk])
    print(f"\n[{label}]")
    print(f"  Doc  NEDS -> Scanned: {doc_sc.get('Scanned', 'N/A'):.4f}  |  Photoed: {doc_sc.get('Photoed', 'N/A'):.4f}")
    print(f"  Tbl  TEDS -> Scanned: {tbl_sc.get('Scanned', 'N/A'):.4f}  |  Photoed: {tbl_sc.get('Photoed', 'N/A'):.4f}")
    print(f"  Avg Token Savings    : {tok_sav:.1f}%")